# W-Flow Visualization Demo

Minimal notebook for checkpoint-based sample generation and visualization.

This notebook uses two repositories and one decoder:
- Code and configs: `https://github.com/hanjq17/W-Flow`
- Checkpoints only: `jiaqihan99/W-Flow` on Hugging Face
- SD-VAE decoder: `stabilityai/sd-vae-ft-mse`

It follows the same logic as `scripts/sample/visualization.sh`:

```bash
python inference_ours.py sample \
  --ckpt "$WFLOW_HF_ROOT/checkpoints/$EXP_NAME/state_$STEPNUM.pt" \
  --config "$CONFIG" \
  --cfg-scale "$cfg" \
  --class-ids "207,829,387,386,360,817,95,927,263,698,483,88" \
  --num-rows 3 \
  --seed 42 \
  --save-path my_grid.png
```

Default example:
- `EXP_NAME = latent_sota_XL_ot`
- `STEPNUM = 00180000`
- `CONFIG = configs/gen/latent_sota_XL_ot_8node.yaml`
- `cfg_scale = 3.0`


In [ ]:
!pip install -q torch==2.5.1 torchvision==0.20.1

In [ ]:
#@title 1. Setup
import os
import sys
from pathlib import Path

# Code/config repo. This is cloned only when the notebook is not already
# running from a checkout that contains inference_ours.py.
CODE_REPO_URL = "https://github.com/hanjq17/W-Flow" #@param {type:"string"}
CODE_REPO_REF = "main" #@param {type:"string"}

# Hugging Face model repo. This repo is expected to contain checkpoints only,
# e.g. checkpoints/latent_sota_XL_ot/state_00180000.pt.
WFLOW_HF_REPO_ID = "jiaqihan99/W-Flow" #@param {type:"string"}
WFLOW_HF_ROOT = "/content/wflow_hf_root" #@param {type:"string"}
DOWNLOAD_WFLOW_CHECKPOINTS = True #@param {type:"boolean"}

# Latent W-Flow checkpoints decode through SD-VAE.
VAE_HF_REPO_ID = "stabilityai/sd-vae-ft-mse" #@param {type:"string"}
VAE_HF_PATH = "/content/sdvae_hf_root" #@param {type:"string"}
DOWNLOAD_VAE_WEIGHTS = True #@param {type:"boolean"}

# torch-fidelity stores its Inception weights here when metrics are imported.
TORCH_HUB_DIR = "/content/torch_hub" #@param {type:"string"}

INSTALL_DEPS = False #@param {type:"boolean"}

def find_repo_dir(start: Path) -> Path | None:
    for candidate in (start, *start.parents):
        if (candidate / "inference_ours.py").is_file():
            return candidate
    return None


def path_is_writable(path: Path) -> bool:
    path = path.expanduser()
    parent = path if path.exists() and path.is_dir() else path.parent
    for candidate in (parent, *parent.parents):
        if candidate.exists():
            return os.access(candidate, os.W_OK)
    return False


def local_writable_dir(path_value: str, fallback_name: str, repo_dir: Path) -> str:
    path = Path(path_value).expanduser()
    if path_is_writable(path):
        return str(path)
    fallback = repo_dir / ".cache" / fallback_name
    print(f"Path {path} is not writable here; using {fallback} instead.")
    return str(fallback)


REPO_DIR = find_repo_dir(Path.cwd())
if REPO_DIR is None:
    REPO_DIR = Path("/content/W-Flow")
    if not path_is_writable(REPO_DIR):
        REPO_DIR = Path.cwd() / "W-Flow"
    if not REPO_DIR.exists():
        print(f"Cloning code from {CODE_REPO_URL} ({CODE_REPO_REF}) to {REPO_DIR} ...")
        get_ipython().system(
            f"git clone --branch {CODE_REPO_REF} --single-branch {CODE_REPO_URL} {REPO_DIR}"
        )
else:
    print(f"Using code repository at {REPO_DIR}")

get_ipython().run_line_magic("cd", str(REPO_DIR))

WFLOW_HF_ROOT = local_writable_dir(WFLOW_HF_ROOT, "wflow_hf_root", REPO_DIR)
VAE_HF_PATH = local_writable_dir(VAE_HF_PATH, "sdvae_hf_root", REPO_DIR)
TORCH_HUB_DIR = local_writable_dir(TORCH_HUB_DIR, "torch_hub", REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

if INSTALL_DEPS:
    print("Installing dependencies listed in requirements.txt using install_env.sh")
    get_ipython().system("bash install_env.sh")

if DOWNLOAD_WFLOW_CHECKPOINTS or DOWNLOAD_VAE_WEIGHTS:
    from huggingface_hub import snapshot_download

if DOWNLOAD_WFLOW_CHECKPOINTS:
    print(f"Downloading checkpoints from {WFLOW_HF_REPO_ID} to {WFLOW_HF_ROOT} ...")
    snapshot_download(
        repo_id=WFLOW_HF_REPO_ID,
        repo_type="model",
        local_dir=WFLOW_HF_ROOT,
    )
else:
    print(f"Skipping checkpoint download. Expecting checkpoints under {WFLOW_HF_ROOT}")

if DOWNLOAD_VAE_WEIGHTS:
    print(f"Downloading SD-VAE from {VAE_HF_REPO_ID} to {VAE_HF_PATH} ...")
    snapshot_download(
        repo_id=VAE_HF_REPO_ID,
        repo_type="model",
        local_dir=VAE_HF_PATH,
    )
else:
    print(f"Skipping VAE download. Expecting SD-VAE weights under {VAE_HF_PATH}")

# Patch the runtime env module before dataset.vae is imported by _load_model.
import utils.env as wflow_env
wflow_env.WFLOW_HF_ROOT = WFLOW_HF_ROOT
wflow_env.VAE_HF_PATH = VAE_HF_PATH
wflow_env.TORCH_HUB_DIR = TORCH_HUB_DIR
wflow_env.IMAGENET_FID_NPZ = str(Path(WFLOW_HF_ROOT) / "stats" / "jit_in256_stats.npz")

print("Setup complete.")


In [ ]:
#@title 2. Helpers
from pathlib import Path

from IPython.display import display
from PIL import Image as PILImage
import torch

from inference_ours import _load_model, run_sample

MODEL_CACHE = {}
print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


def parse_class_ids(text: str) -> list[int]:
    class_ids = [int(part.strip()) for part in text.split(",") if part.strip()]
    if not class_ids:
        raise ValueError("Provide at least one class index.")
    for class_id in class_ids:
        if not 0 <= class_id < 1000:
            raise ValueError(f"class id out of range: {class_id}")
    return class_ids


def resolve_checkpoint(wflow_hf_root: str, exp_name: str, step_num: str) -> Path:
    step_num = str(step_num)
    if step_num.startswith("state_"):
        filename = f"{step_num}.pt" if not step_num.endswith(".pt") else step_num
    else:
        filename = f"state_{step_num.zfill(8)}.pt"
    return Path(wflow_hf_root).expanduser() / "checkpoints" / exp_name / filename


def load_pipeline(ckpt_path: str | Path, config_path: str):
    ckpt_path = Path(ckpt_path).expanduser()
    config_path = Path(config_path).expanduser()
    if not ckpt_path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")
    if not config_path.is_file():
        raise FileNotFoundError(f"Config not found: {config_path}")

    key = (str(ckpt_path), str(config_path))
    if key not in MODEL_CACHE:
        print(f"Loading checkpoint: {key[0]}")
        print(f"Using config: {key[1]}")
        MODEL_CACHE[key] = _load_model(key[0], key[1])
    return MODEL_CACHE[key]


def generate_grid(
    *,
    ckpt_path: str | Path,
    config_path: str,
    class_ids: list[int],
    cfg_scale: float,
    seed: int,
    num_rows: int,
    save_path: str,
):
    model, postprocess_fn, ckpt_step, device = load_pipeline(ckpt_path, config_path)
    run_sample(
        model,
        postprocess_fn,
        class_ids=class_ids,
        cfg_scale=float(cfg_scale),
        seed=int(seed),
        num_rows=int(num_rows),
        save_path=save_path,
        device=device,
    )
    print(f"Checkpoint step: {ckpt_step}")
    display(PILImage.open(save_path))


In [ ]:
#@title 3. Generate
wflow_hf_root = WFLOW_HF_ROOT #@param {type:"string"}
exp_name = "latent_sota_XL_ot" #@param {type:"string"}
step_num = "00180000" #@param {type:"string"}
config = "configs/gen/latent_sota_XL_ot_8node.yaml" #@param {type:"string"}
cfg_scale = 3.0 #@param {type:"number"}
class_ids_text = "207,829,387,386,360,817,95,927,263,698,483,88" #@param {type:"string"}
num_rows = 3 #@param {type:"integer"}
seed = 42 #@param {type:"integer"}
save_path = "my_grid.png" #@param {type:"string"}

ckpt_path = resolve_checkpoint(wflow_hf_root, exp_name, step_num)
class_ids = parse_class_ids(class_ids_text)

print(f"Checkpoint: {ckpt_path}")
print(f"Class IDs: {class_ids}")

generate_grid(
    ckpt_path=ckpt_path,
    config_path=config,
    class_ids=class_ids,
    cfg_scale=cfg_scale,
    seed=seed,
    num_rows=num_rows,
    save_path=save_path,
)
